In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')  # Ensure root directory is in path

from src import LinearModel
from game import PlayerTracker, AdaptiveNPC, simulate_match

# Set global plot style
plt.style.use('seaborn-v0_8-darkgrid')
figsize = (12, 5)

In [ ]:
model = LinearModel(learning_rate=0.01)
tracker = PlayerTracker()
npc = AdaptiveNPC(model)

# Storage for visualization
cost_history = []
weight_history = []
fairness_history = []
match_count = 0

In [ ]:
print("🔧 Generating initial calibration data...")
for _ in range(100):
    skill = np.random.rand()
    result = simulate_match(skill, npc.current_hp, npc.current_attack)
    tracker.record_match(result["win_rate"], result["avg_dmg_taken"], result["time_to_kill"])

X_batch = tracker.get_batch_features()
y_batch = np.full(len(X_batch), 0.5)  # Target: perfect fairness

w, b, costs = model.fit(X_batch, y_batch, iterations=1000)
cost_history.extend(costs)
print(f"✅ Initial training complete. Final cost: {costs[-1]:.6f}")

In [ ]:
print("⚔️ Starting live adaptation phase...")
live_matches = 200

for i in range(live_matches):
    skill = np.random.rand()
    
    # Get features (use last match or default if first)
    if tracker.history:
        features = tracker.get_latest_features()
    else:
        features = np.array([0.5, 100.0, 10.0])
    
    # AI adjusts NPC difficulty
    npc.adjust_difficulty(features)
    
    # Simulate fight
    result = simulate_match(skill, npc.current_hp, npc.current_attack)
    tracker.record_match(result["win_rate"], result["avg_dmg_taken"], result["time_to_kill"])
    
    # Online learning update
    model.partial_fit(features, result["fairness_score"])
    
    # Record metrics
    match_count += 1
    fairness_history.append(result["fairness_score"])
    weight_history.append(model.weights.copy())

print(f"🎯 Completed {live_matches} live matches with real-time adaptation")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=figsize)
fig.suptitle('Adaptive NPC Difficulty Balancer - Learning Progress', fontsize=16, fontweight='bold')

# Plot 1: Batch Training Cost Curve
axes[0, 0].plot(cost_history, 'b-', linewidth=1.5)
axes[0, 0].set_title('Batch Training: MSE Loss Over Iterations')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('MSE Cost')
axes[0, 0].axhline(y=0.125, color='r', linestyle='--', alpha=0.7, label='Perfect Fairness Baseline')
axes[0, 0].legend()

# Plot 2: Live Fairness Score
axes[0, 1].plot(fairness_history, 'g-', linewidth=1.5, alpha=0.8)
axes[0, 1].axhline(y=0.0, color='r', linestyle='--', alpha=0.7, label='Perfect Balance')
axes[0, 1].set_title('Live Matches: Fairness Score (Lower = Better)')
axes[0, 1].set_xlabel('Match Number')
axes[0, 1].set_ylabel('Fairness Score')
axes[0, 1].legend()

# Plot 3: Weight Evolution
weight_array = np.array(weight_history)
for idx, name in enumerate(['Win Rate', 'Avg Dmg Taken', 'Time to Kill']):
    axes[1, 0].plot(weight_array[:, idx], label=name, linewidth=1.5)
axes[1, 0].set_title('Learned Feature Weights Over Time')
axes[1, 0].set_xlabel('Match Number')
axes[1, 0].set_ylabel('Weight Value')
axes[1, 0].legend()

# Plot 4: NPC HP Adaptation
hp_values = [m["avg_dmg_taken"] for m in tracker.history[-live_matches:]]
axes[1, 1].plot(hp_values, 'orange', linewidth=1.5, alpha=0.8)
axes[1, 1].set_title('NPC Damage Taken Trend (Proxy for Difficulty)')
axes[1, 1].set_xlabel('Match Number')
axes[1, 1].set_ylabel('Damage Taken')

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Chart saved as 'learning_curves.png'")